In [ ]:
#SET BASE DIRECTORY
#This notebook expects the GSE135779 data folders (adult_individual_h5ad, GSE135779_RAW, Results, etc.) to sit one level above this Notebooks folder. Update BASE_DIR below if your data lives elsewhere.

import os
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path(BASE_DIR) / "Notebooks"))
from publication_utils import configure_publication_notebook

FIGURE_DIR = configure_publication_notebook(BASE_DIR, "01F_GSE135779_ADULT_SPLITTING")


***3) SPLITTING***

In [ ]:
import scanpy as sc
import sys
sys.path.insert(0, BASE_DIR + '/Notebooks')
from sample_lists import CHILD_HEALTHY_SAMPLES, CHILD_SLE_SAMPLES, ADULT_HEALTHY_SAMPLES, ADULT_SLE_SAMPLES

adata = sc.read_h5ad(
    f"{BASE_DIR}/adult_individual_h5ad/adata_adult_final_liana.h5ad"
)

healthy_samples = ADULT_HEALTHY_SAMPLES

sle_samples = ADULT_SLE_SAMPLES

adata.obs["condition"] = "unknown"
adata.obs.loc[adata.obs["sample"].isin(healthy_samples), "condition"] = "healthy"
adata.obs.loc[adata.obs["sample"].isin(sle_samples), "condition"] = "SLE"

print(adata.obs["condition"].value_counts())
print(adata.obs[["sample", "condition"]].drop_duplicates().sort_values("sample"))

In [ ]:
adata_adult_healthy = adata[adata.obs["condition"] == "healthy"].copy()
adata_adult_sle = adata[adata.obs["condition"] == "SLE"].copy()

print("Healthy:", adata_adult_healthy)
print("SLE:", adata_adult_sle)

In [ ]:
healthy_path = f"{BASE_DIR}/adult_individual_h5ad/adata_adult_healthy_liana.h5ad"
sle_path = f"{BASE_DIR}/adult_individual_h5ad/adata_adult_sle_liana.h5ad"

adata_adult_healthy.write_h5ad(healthy_path)
adata_adult_sle.write_h5ad(sle_path)

print("Saved healthy:", healthy_path)
print("Saved SLE:", sle_path)

***4) SANITY CHECKS***

In [ ]:
import scanpy as sc

adata = sc.read_h5ad(f"{BASE_DIR}/adult_individual_h5ad/adata_adult_final_liana.h5ad")

# check all metadata columns
print(adata.obs.columns.tolist())

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse

# load your final cleaned adult object
adata = sc.read_h5ad(
    f"{BASE_DIR}/adult_individual_h5ad/adata_adult_final_liana.h5ad"
)

print(adata)
print(adata.obs.columns.tolist())

In [ ]:
# required metadata checks
assert "cell_type" in adata.obs.columns, "Missing cell_type column"
assert "sample" in adata.obs.columns, "Missing sample column"

# make cell_type categorical
adata.obs["cell_type"] = adata.obs["cell_type"].astype("category")

# optional but useful
if "gsm_id" in adata.obs.columns:
    adata.obs["gsm_id"] = adata.obs["gsm_id"].astype(str)

print(adata.obs["cell_type"].value_counts())
print(adata.obs["sample"].value_counts())

In [ ]:
# ensure counts layer exists
if "counts" not in adata.layers:
    print("counts layer missing -> creating it from current X")
    adata.layers["counts"] = adata.X.copy()
else:
    print("counts layer already present")

In [ ]:
# quick diagnostic
xmax = adata.X.max() if not sparse.issparse(adata.X) else adata.X.max()
xmin = adata.X.min() if not sparse.issparse(adata.X) else adata.X.min()

print("X min:", xmin)
print("X max:", xmax)

In [ ]:
# no missing cell labels
print("Missing cell_type:", adata.obs["cell_type"].isna().sum())

# enough cells per group
print(adata.obs["cell_type"].value_counts())

# check for duplicate cell names
print("Duplicated obs_names:", adata.obs_names.duplicated().sum())